In [ ]:
!pip uninstall -y langchain
!pip uninstall -y langchain-core
!pip uninstall -y langchain-community
!pip uninstall -y langchain-text-splitters
!pip uninstall -y langchain-huggingface

Found existing installation: langchain 1.3.11
Uninstalling langchain-1.3.11:
  Successfully uninstalled langchain-1.3.11
Found existing installation: langchain-core 1.4.8
Uninstalling langchain-core-1.4.8:
  Successfully uninstalled langchain-core-1.4.8


In [ ]:
!pip install -U \
langchain \
langchain-community \
langchain-core \
langchain-text-splitters \
langchain-huggingface \
sentence-transformers \
faiss-cpu \
pypdf \
transformers \
accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

/tmp/ipykernel_1877/2671165056.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving knowledge2.pdf to knowledge2.pdf


In [ ]:
import os

print(os.listdir())

['.config', 'knowledge2.pdf', 'sample_data']


In [ ]:
loader = PyPDFLoader("knowledge2.pdf")
documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 943


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print("Total Chunks:", len(docs))

Total Chunks: 5809


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings Loaded Successfully")

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings Loaded Successfully


In [ ]:
print(len(documents))

In [ ]:
print(len(docs))

In [ ]:
print(documents[0].page_content[:500])

In [ ]:
print(repr(documents[0].page_content))

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

print("Vector Store Created Successfully!")

In [ ]:
vectorstore.save_local("vectorstore")
print("Vector Store Saved Successfully!")

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="gpt2"
)

print("Pipeline Working Successfully!")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 Loaded Successfully!")

In [ ]:
question = "What is Artificial Intelligence?"

inputs = tokenizer(question, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=100)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke("What is Artificial Intelligence?")

print("Retrieved Chunks:", len(results))

print("\nFirst Retrieved Chunk:\n")
print(results[0].page_content)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model Loaded Successfully!")

In [ ]:
question = "Explain Artificial Intelligence in simple words."

inputs = tokenizer(question, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=100)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

In [ ]:
question = "What is Artificial Intelligence?"

docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{question}

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)

outputs = model.generate(**inputs, max_new_tokens=200)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

In [ ]:
print(len(docs))